# TCC II | Comparação de classificadores com histórico de leituras

Árvore de Decisão, SVM com núcleo RBF e Random Forest classificam a condição
do módulo no instante avaliado como operação normal, acúmulo de poeira
ou sombreamento. Todos recebem os mesmos 15 atributos e são avaliados nas
mesmas partições cronológicas. A métrica principal é o F1 macro.

No Colab, use **Ambiente de execução > Executar tudo**. O experimento usa
CPU e gera os próprios dados. Ao final, baixe **resultados_tcc_comparacao_com_historico.zip**
pelo painel de arquivos.


In [ ]:
%pip install -q pvlib==0.16.1 scikit-learn==1.6.1


## 1. Gerar dados simulados

A simulação usa um módulo fotovoltaico em Quixadá (CE) e gera 120 dias em passos de cinco minutos. A sujeira cresce entre limpezas. Eventos breves de sombra são aproximados pela redução da irradiância efetiva do módulo e têm prioridade no rótulo quando ocorrem junto com sujeira. O rótulo de sujeira é aplicado a partir de perdas acima de 12% na tendência simulada.

In [ ]:
import numpy as np
import pandas as pd
import pvlib
from pvlib.location import Location
from pvlib.pvsystem import retrieve_sam, calcparams_cec, singlediode
from pvlib.temperature import faiman

# ---------------------------------------------------------------------------
# 1. CONFIGURAÇÃO GERAL
# ---------------------------------------------------------------------------

LATITUDE = -4.97      # Quixadá-CE, ajuste para sua localização real
LONGITUDE = -39.02
ALTITUDE = 200
TZ = "America/Fortaleza"

TILT = 20          # inclinação do painel (graus)
AZIMUTH = 0        # 0 = voltado para o norte geográfico (hemisfério sul); ajuste conforme instalação

DIAS_SIMULADOS = 120     # quantidade de dias sintéticos a gerar
FREQ = "5min"            # resolução temporal da série
DATA_INICIO = "2025-01-01"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---------------------------------------------------------------------------
# 2. MÓDULO FOTOVOLTAICO (banco de dados CEC embutido no pvlib)
# ---------------------------------------------------------------------------

cec_modules = retrieve_sam("CECMod")
MODULO_CEC = "Canadian_Solar_Inc__CS6X_300M"   # troque pelo modelo real do seu painel, se disponível
if MODULO_CEC not in cec_modules.columns:
    # fallback: pega o primeiro módulo Canadian_Solar disponível na base
    MODULO_CEC = [c for c in cec_modules.columns if "Canadian_Solar" in c][0]

modulo = cec_modules[MODULO_CEC]

# Parâmetros térmicos do módulo (modelo Faiman - simples e robusto, exige só vento)
U0, U1 = 25.0, 6.84   # valores típicos para módulo em vidro/célula/polímero

# ---------------------------------------------------------------------------
# 3. GERAÇÃO DO CLIMA BASE (irradiância de céu claro + variação sintética)
# ---------------------------------------------------------------------------

site = Location(LATITUDE, LONGITUDE, tz=TZ, altitude=ALTITUDE)

times = pd.date_range(
    start=DATA_INICIO, periods=int(pd.Timedelta(days=DIAS_SIMULADOS) / pd.Timedelta(FREQ)),
    freq=FREQ, tz=TZ
)

# irradiância de céu claro (Ineichen) como base "ideal"
clearsky = site.get_clearsky(times, model="ineichen")

# posição solar (necessária para converter GHI/DNI/DHI em irradiância no plano do painel)
solpos = site.get_solarposition(times)

poa = pvlib.irradiance.get_total_irradiance(
    surface_tilt=TILT,
    surface_azimuth=AZIMUTH,
    solar_zenith=solpos["apparent_zenith"],
    solar_azimuth=solpos["azimuth"],
    dni=clearsky["dni"],
    ghi=clearsky["ghi"],
    dhi=clearsky["dhi"],
)
poa_global = poa["poa_global"].clip(lower=0)

# ruído de nebulosidade realista (fator multiplicativo suave, sempre >= 0)
ruido_nuvem = np.clip(np.random.normal(loc=1.0, scale=0.05, size=len(times)), 0.6, 1.05)
poa_global = poa_global * ruido_nuvem

# temperatura ambiente sintética: senoide diária com variação dia-a-dia + ruído
hora_do_dia = times.hour + times.minute / 60
temp_base = 26 + 6 * np.sin((hora_do_dia - 9) / 24 * 2 * np.pi)
temp_ambiente = temp_base + np.random.normal(0, 0.8, size=len(times))

vento = np.clip(np.random.normal(2.5, 1.0, size=len(times)), 0.2, None)  # m/s

# ---------------------------------------------------------------------------
# 4. DEFINIÇÃO DOS ESTADOS DO PAINEL (processo físico contínuo, não blocos fixos)
# ---------------------------------------------------------------------------
# Modelagem mais realista de como as falhas acontecem na prática:
#
#   - SOILING: processo LENTO e CUMULATIVO. A sujeira vai se acumulando dia
#     após dia (perda de irradiância crescente) até uma limpeza (manutenção),
#     que zera o acúmulo e reinicia o ciclo. Por isso ela permanece ativa ao
#     longo de VÁRIOS DIAS seguidos, não é sorteada por bloco isolado.
#
#   - SOMBREAMENTO: evento CURTO e PONTUAL (nuvem passando, poste/árvore
#     projetando sombra por causa do ângulo baixo do sol de manhã/fim de
#     tarde, objeto temporário). Dura minutos, pode ocorrer VÁRIAS VEZES no
#     mesmo dia, com maior probabilidade perto do nascer/pôr do sol.
#
# Isso permite que um único dia tenha: sujeira de fundo (se estiver dentro de
# um ciclo de acúmulo) + um ou mais eventos de sombreamento sobrepostos.

n = len(times)
dia_indice = ((times.normalize() - times.normalize()[0]) / pd.Timedelta(days=1)).astype(int)
tempo_continuo_dias = (times - times[0]) / pd.Timedelta(days=1)   # tempo decorrido, em dias, CONTÍNUO (não em degraus)

# --- 4a. Sujeira (soiling): acumula linearmente e de forma contínua entre limpezas ---
sujeira_fraction = np.zeros(n)
ciclo_limpeza = np.zeros(n, dtype=int)
dia_atual = 0
while dia_atual < DIAS_SIMULADOS:
    duracao_ciclo = min(np.random.randint(6, 15), DIAS_SIMULADOS - dia_atual)  # dias até a próxima limpeza
    perda_max_ciclo = np.random.uniform(0.15, 0.40)  # perda de irradiância no pior dia do ciclo (antes de limpar)

    mask_ciclo = (dia_indice >= dia_atual) & (dia_indice < dia_atual + duracao_ciclo)
    ciclo_limpeza[mask_ciclo] = dia_atual
    tempo_relativo = tempo_continuo_dias[mask_ciclo] - dia_atual   # 0 -> duracao_ciclo, CONTÍNUO amostra a amostra
    progresso = tempo_relativo / duracao_ciclo   # 0 -> quase 1, sobe suavemente (não em degraus diários)
    sujeira_fraction[mask_ciclo] = perda_max_ciclo * progresso

    dia_atual += duracao_ciclo

sujeira_fraction_label = sujeira_fraction.copy()   # tendência "limpa", usada só para o rótulo
sujeira_fraction = np.clip(sujeira_fraction + np.random.normal(0, 0.008, size=n), 0, None)  # com ruído, usada na física
LIMIAR_SOILING_VISIVEL = 0.12   # só rotula como falha quando a perda já é inequívoca (>12%), bem acima do ruído de nuvem

# --- 4b. Sombreamento: eventos curtos e pontuais, com viés para manhã/fim de tarde ---
sombra_fator = np.ones(n)
sombra_ativo = np.zeros(n, dtype=bool)

media_eventos_por_dia = 2.5
n_eventos = np.random.poisson(media_eventos_por_dia * DIAS_SIMULADOS)

for _ in range(n_eventos):
    dia = np.random.randint(0, DIAS_SIMULADOS)
    # horário do evento: bimodal, concentrado perto do nascer/pôr do sol (baixo ângulo solar)
    if np.random.rand() < 0.5:
        hora = np.clip(np.random.normal(6.5, 1.0), 5.0, 9.5)
    else:
        hora = np.clip(np.random.normal(16.5, 1.2), 13.5, 18.5)

    inicio = times.normalize()[0] + pd.Timedelta(days=int(dia), hours=float(hora))
    duracao_evento = pd.Timedelta(minutes=np.random.uniform(5, 40))
    fim = inicio + duracao_evento

    mask_evento = (times >= inicio) & (times < fim)
    if not mask_evento.any():
        continue

    fator_base = np.random.uniform(0.30, 0.65)
    ruido_sombra = np.random.normal(1.0, 0.08, size=mask_evento.sum())
    sombra_fator[mask_evento] = np.minimum(sombra_fator[mask_evento], fator_base * ruido_sombra)
    sombra_ativo[mask_evento] = True

sombra_fator = np.clip(sombra_fator, 0, 1)

# --- 4c. Combina os dois efeitos na irradiância efetiva e define o rótulo do estado ---
irradiancia_efetiva = poa_global.values * (1 - sujeira_fraction) * sombra_fator
irradiancia_efetiva = np.clip(irradiancia_efetiva, 0, None)

estados = np.full(n, "normal", dtype=object)
estados[sujeira_fraction_label > LIMIAR_SOILING_VISIVEL] = "soiling"
estados[sombra_ativo] = "sombreamento"   # evento de sombreamento tem prioridade sobre o rótulo de soiling de fundo

# ---------------------------------------------------------------------------
# 5. TEMPERATURA DE CÉLULA (modelo Faiman)
# ---------------------------------------------------------------------------

temp_celula = faiman(
    poa_global=irradiancia_efetiva,
    temp_air=temp_ambiente.values,
    wind_speed=vento,
    u0=U0, u1=U1,
)

# ---------------------------------------------------------------------------
# 6. MODELO DE DIODO ÚNICO -> TENSÃO E CORRENTE NO PONTO DE MÁXIMA POTÊNCIA
# ---------------------------------------------------------------------------
# Assume-se que o sistema tem MPPT (como na prática), então o ponto de
# operação lido pelos sensores é (Vmp, Imp).

params = calcparams_cec(
    effective_irradiance=irradiancia_efetiva,
    temp_cell=temp_celula,
    alpha_sc=modulo["alpha_sc"],
    a_ref=modulo["a_ref"],
    I_L_ref=modulo["I_L_ref"],
    I_o_ref=modulo["I_o_ref"],
    R_sh_ref=modulo["R_sh_ref"],
    R_s=modulo["R_s"],
    Adjust=modulo["Adjust"],
)

iv = singlediode(*params, method="lambertw")

tensao = iv["v_mp"]
corrente = iv["i_mp"]
potencia = iv["p_mp"]

# quando não há luz suficiente (noite), zera tudo e marca como "normal" (painel desligado/repouso)
sem_luz = irradiancia_efetiva < 5
tensao = np.where(sem_luz, 0.0, tensao)
corrente = np.where(sem_luz, 0.0, corrente)
potencia = np.where(sem_luz, 0.0, potencia)
estados = np.where(sem_luz, "normal", estados)

# ---------------------------------------------------------------------------
# 7. RUÍDO DE SENSOR (para simular leituras reais de sensores V/I/T embarcados)
# ---------------------------------------------------------------------------

tensao_sensor = np.clip(tensao + np.random.normal(0, 0.15, size=n), 0, None)
corrente_sensor = np.clip(corrente + np.random.normal(0, 0.05, size=n), 0, None)
temp_celula_sensor = temp_celula + np.random.normal(0, 0.3, size=n)

# ---------------------------------------------------------------------------
# 8. MONTAGEM DO DATASET FINAL
# ---------------------------------------------------------------------------

df = pd.DataFrame({
    "timestamp": times,
    "ghi": clearsky["ghi"].values,
    "poa_global": poa_global.values,
    "poa_ceu_limpo": poa["poa_global"].clip(lower=0).to_numpy(),
    "ciclo_limpeza": ciclo_limpeza,
    "irradiancia_efetiva": irradiancia_efetiva,
    "temp_ambiente": temp_ambiente.values,
    "temp_celula": temp_celula_sensor,
    "tensao": tensao_sensor,
    "corrente": corrente_sensor,
    "potencia": tensao_sensor * corrente_sensor,
    "estado": estados,
})

print(f"{len(df):,} amostras simuladas em {DIAS_SIMULADOS} dias.")


## 2. Construir os atributos e separar os ciclos

Os três modelos recebem três leituras atuais e 12 atributos de histórico.
Para cada grandeza (tensão, corrente e temperatura), calculamos duas médias
das leituras anteriores (15 e 30 minutos) e duas diferenças (leitura atual
menos a leitura de 15 e de 30 minutos atrás). O alvo é a classe do instante
atual.

As janelas seguem a grade de cinco minutos antes da seleção das amostras e
reiniciam a cada dia. Aplicamos o mesmo recorte do notebook do Random Forest
com histórico: irradiância de céu limpo estimada de pelo menos 100 W/m² e
histórico completo. A estimativa serve apenas para selecionar os instantes
avaliados. O último quarto dos ciclos de limpeza é reservado para teste, e
três divisões cronológicas nos ciclos anteriores são usadas na validação.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import TimeSeriesSplit


BASE = ["tensao", "corrente", "temp_celula"]
CLASSES = ["normal", "soiling", "sombreamento"]
ROTULOS = ["Normal", "Poeira", "Sombreamento"]
JANELAS = [(15, 3), (30, 6)]


def construir_atributos(leituras):
    TEMPORAIS = []

    serie = (
        leituras.set_index("timestamp")
        .sort_index()
        .asfreq("5min")
    )
    dias = serie.index.normalize()

    for coluna in BASE:
        por_dia = serie[coluna].groupby(dias)
        for minutos, n_leituras in JANELAS:
            media = f"{coluna}_media_{minutos}min"
            variacao = f"{coluna}_variacao_{minutos}min"

            # Usa as leituras anteriores a t, em ordem cronológica.
            serie[media] = por_dia.transform(
                lambda grupo: grupo.shift(1).rolling(
                    n_leituras, min_periods=n_leituras
                ).mean()
            )
            serie[variacao] = serie[coluna] - por_dia.shift(n_leituras)
            TEMPORAIS.extend([media, variacao])
    return serie, BASE + TEMPORAIS


serie, ATRIBUTOS = construir_atributos(df)
dados_diurnos = serie.loc[serie["poa_ceu_limpo"] >= 100]
dados = dados_diurnos.dropna(subset=ATRIBUTOS + ["estado"]).reset_index()
print(f"Amostras após o filtro de irradiância: {len(dados_diurnos):,}")
print(f"Amostras com histórico completo: {len(dados):,}")

ciclos = dados["ciclo_limpeza"].unique()
n_teste = int(np.ceil(len(ciclos) / 4))
if len(ciclos) - n_teste < 4:
    raise ValueError("Gere mais ciclos para obter três rodadas de validação.")

ciclos_ajuste = ciclos[:-n_teste]
ciclos_teste = ciclos[-n_teste:]
ajuste = dados[dados["ciclo_limpeza"].isin(ciclos_ajuste)].reset_index(drop=True)
teste = dados[dados["ciclo_limpeza"].isin(ciclos_teste)].reset_index(drop=True)

X_ajuste, y_ajuste = ajuste[ATRIBUTOS], ajuste["estado"]
X_teste, y_teste = teste[ATRIBUTOS], teste["estado"]
print(f"Atributos por amostra: {len(ATRIBUTOS)}")

cv = []
particoes = {"Ajuste completo": ajuste, "Teste final": teste}

for rodada, (treino, validacao) in enumerate(
    TimeSeriesSplit(n_splits=3).split(ciclos_ajuste), start=1
):
    idx_treino = np.flatnonzero(
        ajuste["ciclo_limpeza"].isin(ciclos_ajuste[treino]).to_numpy()
    )
    idx_validacao = np.flatnonzero(
        ajuste["ciclo_limpeza"].isin(ciclos_ajuste[validacao]).to_numpy()
    )
    cv.append((idx_treino, idx_validacao))
    particoes[f"Treino {rodada}"] = ajuste.iloc[idx_treino]
    particoes[f"Validação {rodada}"] = ajuste.iloc[idx_validacao]

resumo_particoes = []
for nome, parte in particoes.items():
    contagens = parte["estado"].value_counts().reindex(CLASSES, fill_value=0)
    resumo_particoes.append({
        "Partição": nome,
        "Início": parte["timestamp"].min(),
        "Fim": parte["timestamp"].max(),
        "Ciclos": parte["ciclo_limpeza"].nunique(),
        "Amostras": len(parte),
        **contagens.to_dict(),
    })

tabela_particoes = pd.DataFrame(resumo_particoes)
display(tabela_particoes)
if (tabela_particoes[CLASSES] == 0).any().any():
    raise ValueError("Há uma classe ausente em alguma partição. Gere mais dias.")
if not np.isfinite(dados[ATRIBUTOS].to_numpy()).all():
    raise ValueError("Foram encontrados valores ausentes ou infinitos nos atributos.")


## 3. Ajustar e avaliar os três classificadores

Cada modelo é ajustado usando a mesma divisão em três rodadas de validação
e recebe os mesmos atributos. O F1 macro médio orienta a escolha dos
hiperparâmetros. Para a SVM, a padronização de cada um dos 15 atributos é
ajustada só com os dados de treino de cada rodada. Após a seleção, cada
classificador é reajustado no conjunto de desenvolvimento e avaliado nos
mesmos ciclos reservados para o teste final.


In [ ]:
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

modelos = {
    "Árvore de Decisão": (
        DecisionTreeClassifier(class_weight="balanced", random_state=42),
        {"max_depth": [4, 8, 12], "min_samples_leaf": [5, 20]},
    ),
    "SVM (RBF)": (
        make_pipeline(StandardScaler(), SVC(kernel="rbf", class_weight="balanced")),
        {"svc__C": [0.1, 1, 10], "svc__gamma": ["scale", 0.1]},
    ),
    "Random Forest": (
        RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=42
        ),
        {"max_depth": [6, 12, None], "min_samples_leaf": [5, 20]},
    ),
}

resultados = []
metricas_por_classe = []
previsoes = {}

for nome, (modelo, grade) in modelos.items():
    print(f"Ajustando {nome}...", flush=True)
    busca = GridSearchCV(
        modelo, grade, scoring="f1_macro", cv=cv, n_jobs=2, refit=True,
        error_score="raise",
    )
    busca.fit(X_ajuste, y_ajuste)
    pred = busca.predict(X_teste)
    previsoes[nome] = pred

    relatorio = classification_report(
        y_teste, pred, labels=CLASSES, output_dict=True, zero_division=0
    )
    resultados.append({
        "Classificador": nome,
        "F1 macro (validação)": busca.best_score_,
        "F1 macro (teste)": f1_score(
            y_teste, pred, labels=CLASSES, average="macro", zero_division=0
        ),
        "Acurácia (teste)": accuracy_score(y_teste, pred),
        "Poeira prevista como normal": int(
            (y_teste.eq("soiling") & (pred == "normal")).sum()
        ),
        "Hiperparâmetros": json.dumps(busca.best_params_, ensure_ascii=False),
    })

    for classe, rotulo in zip(CLASSES, ROTULOS):
        metricas = relatorio[classe]
        metricas_por_classe.append({
            "Classificador": nome,
            "Classe": rotulo,
            "Precisão": metricas["precision"],
            "Revocação": metricas["recall"],
            "F1": metricas["f1-score"],
            "Amostras": int(metricas["support"]),
        })

tabela = pd.DataFrame(resultados).sort_values("F1 macro (teste)", ascending=False)
tabela_classes = pd.DataFrame(metricas_por_classe)
display(tabela.round(3))
display(tabela_classes.round(3))


## 4. Gerar as matrizes de confusão e exportar

As linhas mostram a classe atribuída pelo simulador e as colunas mostram a classe prevista. A célula exporta a comparação geral, as métricas por classe, as partições, o PDF com as matrizes e as versões das bibliotecas usadas.

In [ ]:
import platform
import zipfile
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

saida = Path("resultados_tcc_comparacao_com_historico")
saida.mkdir(exist_ok=True)
arquivos = [
    saida / "comparacao_classificadores.csv",
    saida / "metricas_por_classe.csv",
    saida / "particoes_temporais.csv",
    saida / "matrizes_confusao.pdf",
    saida / "versoes.txt",
]
tabela.to_csv(arquivos[0], index=False)
tabela_classes.to_csv(arquivos[1], index=False)
tabela_particoes.to_csv(arquivos[2], index=False)

fig, eixos = plt.subplots(1, 3, figsize=(15, 4.5), layout="constrained")
fig.suptitle("Classificadores com histórico de leituras")
for eixo, (nome, pred) in zip(eixos, previsoes.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_teste, pred, labels=CLASSES, display_labels=ROTULOS,
        cmap="Blues", colorbar=False, ax=eixo,
    )
    eixo.set_title(nome)
    eixo.set_xlabel("Classe prevista")
    eixo.set_ylabel("Classe simulada")

fig.savefig(arquivos[3], bbox_inches="tight")
plt.show()

pacotes = ("pvlib", "scikit-learn", "numpy", "pandas", "scipy", "matplotlib")
versoes = [f"Python {platform.python_version()}"]
versoes.extend(f"{pacote} {version(pacote)}" for pacote in pacotes)
arquivos[4].write_text("\n".join(versoes) + "\n", encoding="utf-8")

zip_path = Path("resultados_tcc_comparacao_com_historico.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zip_saida:
    for arquivo in arquivos:
        zip_saida.write(arquivo, arcname=arquivo.name)
print(f"Baixe {zip_path.name} pelo painel de arquivos do Colab.")


## Como interpretar a comparação

Compare o F1 macro no teste e confira a precisão, a revocação e o F1 por
classe. Use as matrizes para identificar quais condições foram confundidas.
A tabela geral também mostra quantas amostras de poeira cada modelo
classificou como operação normal.
